# Fox scenes: make the long video
Set `SCENE` and `TYPE`, then press the play button on the cell below. It builds the video in Google's cloud and saves it into your Google Drive, so nothing is stored on your phone or computer.

In [ ]:
# Builds a fox scene video and saves it straight into your Google Drive.
# Pick the scene and the type, press the play button on the left, allow Google Drive access, and wait.
# It enlarges the three-minute loop to 2560x1440 for YouTube, mixes two hours of sound and joins everything.
# Expect roughly 20 to 30 minutes for a 2-hour lo-fi video and 25 to 35 for a 3-hour ambience video. The 3-hour file is
# about 9 GB, so keep that much free in Google Drive.
# Lo-fi uses the tracks made by make-music.ipynb if they are in your Drive (My Drive / Cozy Scenes / music / <scene>);
# without them it composes its own simpler music instead.

SCENE = "toymaker"               # "toymaker" (Christmas Snowfall at the Toymaker's Workshop) or "stationmaster" (Blizzard at the Stationmaster's Waiting Room)
TYPE = "lofi"                    # "lofi" (music over the scene's sounds) or "ambience" (the scene's sounds only, no music)
DRIVE_FOLDER = "Cozy Scenes"     # folder in My Drive where the video is saved

import glob, json, os, shutil, subprocess, urllib.request
from google.colab import drive
drive.mount("/content/drive")

BASE = f"https://raw.githubusercontent.com/codingbythy/claude/refs/heads/claude/inspiring-faraday-20xabu/fox-scenes/{SCENE}/video-kit/"
work = f"/content/{SCENE}"
os.makedirs(work, exist_ok=True)
os.chdir(work)
kit = json.load(urllib.request.urlopen(BASE + "kit.json"))
for name in ["kit.json"] + kit["files"]:
    print("Downloading", name)
    subprocess.run(["wget", "-q", "-O", name, BASE + name], check=True)

hours = kit["mixes"][TYPE].get("hours", kit["hours"])   # each type has its own length: lo-fi 2 hours, ambience 3
out = f"{kit['slug']}-{TYPE}-{hours}h.mp4"      # the length in the name comes from the kit, the same number the thumbnail badge uses
print("Building", out)
music = []
tracks = sorted(glob.glob(f"/content/drive/MyDrive/{DRIVE_FOLDER}/music/{SCENE}/*.flac")) if TYPE == "lofi" else []
if len(tracks) >= 8:
    local = f"/content/music-{SCENE}"                  # copied off Drive first: reading it track by track from Drive is slow
    shutil.rmtree(local, ignore_errors=True); os.makedirs(local)
    for t in tracks:
        shutil.copy(t, local)
    music = ["--music-dir", local]
    print(f"Using the {len(tracks)} tracks in My Drive / {DRIVE_FOLDER} / music / {SCENE}")
elif TYPE == "lofi":
    print("No tracks from make-music.ipynb found in your Drive, so the music will be composed here instead.")
fire = []
if kit.get("fire"):          # the fire's sound files live in your Drive, not in the public kit (myNoise, used with permission)
    stems = f"/content/drive/MyDrive/{DRIVE_FOLDER}/{kit['fire']['drive_folder']}"
    need = [band[s] + ".ogg" for band in kit["fire"]["bands"] for s in ("a", "b")]
    missing = [n for n in need if not os.path.exists(f"{stems}/{n}")]
    if missing:
        raise SystemExit(f"The fire needs {', '.join(missing)} in My Drive / {DRIVE_FOLDER} / {kit['fire']['drive_folder']}.")
    local = f"/content/fire-{SCENE}"; os.makedirs(local, exist_ok=True)
    for n in need:
        shutil.copy(f"{stems}/{n}", local)
    fire = ["--fire-dir", local]
    print(f"Fire: {kit['fire']['name']}")
subprocess.run(["python3", "build.py", out, "--type", TYPE] + music + fire, check=True)

dest = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(dest, exist_ok=True)
print(f"Copying to Google Drive ({os.path.getsize(out) / 1e9:.1f} GB)...")
shutil.copy(out, dest)
drive.flush_and_unmount()
print(f"Done: My Drive / {DRIVE_FOLDER} / {out}")
